## 전처리시 주의사항
#### 1. 데이터의 양이 너무 크면(10만건 이상) 커널이 죽을수있음
#### 2. 토큰이 많아도 커널이 죽을수있음
#### 3. 커널 죽을시 코랩으로
#### 4. 코사인 유사도/빈도수로 토큰처리 코드는 무조건 통합데이터로 돌릴것

## 라이브러리

In [1]:
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm
import time, pickle, os
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from konlpy.tag import Okt
from collections import Counter
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer
#from gensim.models.doc2vec import TaggedDocument
#from gensim.models import Doc2Vec
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics.cluster import silhouette_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD

import html

In [2]:
#path = "./data/최종_통합본.xls"  # 확장자가 .xls라도 실제로는 CSV일 수 있음, 경로에 맞춰서 설정
#df = pd.read_csv(path, encoding="utf-8-sig")

df = pd.read_csv('./data/통합본/통합본.csv',encoding = 'utf-8-sig')

In [3]:
df.shape

(166191, 67)

In [36]:
df.head()

,content,date,url,period_start,period_end,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,...,Unnamed: 58,Unnamed: 59,Unnamed: 60,Unnamed: 61,Unnamed: 62,contents,nouns,author,text,Unnamed: 0
0,#성인ADHD\n#성인ADHD자가진단\n나는 왜 이렇게 집중을 못할까?\n나는 왜 ...,2023. 9. 1. 10:25,https://blog.naver.com/krnfqhr/223199403171,20230822,20230904,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,성인 ADHD 테스트 방법 원인 유전 있을까요?\n성인ADHD\n성인 ADHD는 인...,2023. 8. 29. 16:13,https://blog.naver.com/sominclinic/223196717378,20230822,20230904,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"성인 ADHD 원인, 증상 그리고 자가진단 방법\n성인 ADHD는 주의력 결핍 과잉...",2023. 8. 25. 15:47,https://blog.naver.com/yourspecialist/22319330...,20230822,20230904,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,#김해성인ADHD상담#김해심리상담센터#김해우울증상담#김해조울증상담#김해불안증상담#김...,2023. 8. 24. 11:50,https://blog.naver.com/bori32/223192132796,20230822,20230904,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,주위에 보면 꼭 주의력이 안 좋은 친구들이 하나둘씩 있죠? 친구뿐만 아니라 가족들 ...,2023. 8. 22. 19:53,https://blog.naver.com/nasaco2/223190628029,20230822,20230904,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 컬럼 삭제

In [4]:
## Unnamed가 있을경우 실행해서 삭제
def columns_remove(df):
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    return df
    
df = columns_remove(df)

In [5]:
df.shape

(166191, 9)

In [42]:
df.columns

Index(['content', 'date', 'url', 'period_start', 'period_end', 'contents',
       'nouns', 'author', 'text'],
      dtype='object')

In [46]:
df['contents']

0                                                       NaN
1                                                       NaN
2                                                       NaN
3                                                       NaN
4                                                       NaN
                                ...                        
166186    저는 스무살이고, 지인은 고등학생이에요. 지인은 가정사가 좀 어둡고, 지금도 그닥 ...
166187    사실 최근에 문제가 해결됬습니다만... 걱정이 되어서 글 적어봅니다.근 몇달간 아침...
166188    조금씩 회복중이긴한데 얼른 고치고 싶습니다. 무기력함때문에 아무것도 못하고 있고 너...
166189                                             이 무엇인가요?
166190    요즘들어 만사가 다 귀찮고아무것도 하기싫어요.심지어 씻는것도 귀찮아서주말에 외출하면...
Name: contents, Length: 166191, dtype: object

## 피처 엔지니어링   

In [5]:
#분석에 사용할 컬럼 만들기
df['articles'] = (
                  df['content'].fillna("").astype(str) + " " +
                  df['contents'].fillna("").astype(str) + " " +
                  df['text'].fillna("").astype(str)
                 )

#원문 데이터를 확인해야하니 고려해서 컬럼 선정

df = df[['date','content','contents','text','articles']]
#df = df[['date','title','content','reviews','articles']]

In [58]:
df

,date,content,contents,text,articles
0,2023. 9. 1. 10:25,#성인ADHD\n#성인ADHD자가진단\n나는 왜 이렇게 집중을 못할까?\n나는 왜 ...,NaN,NaN,#성인ADHD\n#성인ADHD자가진단\n나는 왜 이렇게 집중을 못할까?\n나는 왜 ...
1,2023. 8. 29. 16:13,성인 ADHD 테스트 방법 원인 유전 있을까요?\n성인ADHD\n성인 ADHD는 인...,NaN,NaN,성인 ADHD 테스트 방법 원인 유전 있을까요?\n성인ADHD\n성인 ADHD는 인...
2,2023. 8. 25. 15:47,"성인 ADHD 원인, 증상 그리고 자가진단 방법\n성인 ADHD는 주의력 결핍 과잉...",NaN,NaN,"성인 ADHD 원인, 증상 그리고 자가진단 방법\n성인 ADHD는 주의력 결핍 과잉..."
3,2023. 8. 24. 11:50,#김해성인ADHD상담#김해심리상담센터#김해우울증상담#김해조울증상담#김해불안증상담#김...,NaN,NaN,#김해성인ADHD상담#김해심리상담센터#김해우울증상담#김해조울증상담#김해불안증상담#김...
4,2023. 8. 22. 19:53,주위에 보면 꼭 주의력이 안 좋은 친구들이 하나둘씩 있죠? 친구뿐만 아니라 가족들 ...,NaN,NaN,주위에 보면 꼭 주의력이 안 좋은 친구들이 하나둘씩 있죠? 친구뿐만 아니라 가족들 ...
...,...,...,...,...,...
166186,2024.09.19,NaN,"저는 스무살이고, 지인은 고등학생이에요. 지인은 가정사가 좀 어둡고, 지금도 그닥 ...",NaN,"저는 스무살이고, 지인은 고등학생이에요. 지인은 가정사가 좀 어둡고, 지금도 그닥..."
166187,2024.09.27,NaN,사실 최근에 문제가 해결됬습니다만... 걱정이 되어서 글 적어봅니다.근 몇달간 아침...,NaN,사실 최근에 문제가 해결됬습니다만... 걱정이 되어서 글 적어봅니다.근 몇달간 아...
166188,2024.10.21,NaN,조금씩 회복중이긴한데 얼른 고치고 싶습니다. 무기력함때문에 아무것도 못하고 있고 너...,NaN,조금씩 회복중이긴한데 얼른 고치고 싶습니다. 무기력함때문에 아무것도 못하고 있고 ...
166189,2024.10.12,NaN,이 무엇인가요?,NaN,이 무엇인가요?


In [6]:
df2 = df.copy()

## 광고 제거

In [33]:
## 강력한 버전

In [60]:
tqdm.pandas()
# 1️⃣ 광고 패턴 정의
AD_PATTERNS = [
    r"카톡", r"상담", r"구매\s*문의", r"도서" ,r"썰" ,r"영어" ,r"치아",r"자소서", r"과외",
    r"최저가", r"특가", r"할인", r"치과" ,r"검진" , r"기술" ,r"임플란트",r"복싱", r"캠프",
    r"문의\s*주세요", r"클릭", r"육아" , r"독서" ,r"사마귀" ,r"내시경", r"휘트니스", r"문의주세요",r"전화주세요",
    r"쿠폰", r"공동구매", r"체험단" , r"비뇨기과" ,r"구강" ,r"접종" , r"피트니스",
    r"이벤트", r"광고", r"협찬", r"제공받", r"지원받", r"후원받", r"sponsored", r"\bAD\b", r"PR",
    r"구매링크", r"쇼핑몰", r"프로모션", r"이벤트", r"단독", r"특별가",r"전화\s*주세요",r"연락\s*주세요",
    r"무료체험", r"무료상담", r"문의하기", r"빠른상담", r"전화상담", r"카카오톡", r"톡톡",
    r"피부과", r"한의원", r"시술", r"보톡스", r"필러", r"여드름", r"탈모", r"모발이식",
    r"투자", r"대출", r"부동산", r"분양", r"중개", r"청약",
    r"과외", r"레슨", r"강의", r"클래스", r"교육", r"모집", r"채용", r"스터디",
    r"안내드립니다", r"진행하고 있습니다", r"빠르게 연락", r"상세내용",
]
AD_REGEX = re.compile("|".join(AD_PATTERNS))

# 2️⃣ 광고 포함 여부 판별 함수
def is_notice(text: str) -> bool:
    """광고성 문구가 포함되어 있으면 True 반환"""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    return bool(AD_REGEX.search(text))

# 3️⃣ 광고글 판별 후 제거
df["is_notice"] = df["articles"].progress_apply(is_notice)  # ✅ 새 컬럼으로 추가
df = df[df["is_notice"] == False].reset_index(drop=True)

# (선택) 컬럼 정리
df = df.drop(columns=["is_notice"])
#df = df.drop(columns=['Unnamed: 0'])

  0%|          | 0/166191 [00:00<?, ?it/s]

In [61]:
df.shape

(95885, 5)

In [42]:
tqdm.pandas()
# 1️⃣ 광고 패턴 정의
# 띄어쓰기 허용
# 0️⃣ ADHD 관련 키워드 (화이트리스트용)
WHITE_PATTERNS = [
    r"ADHD",r"주의력\s*결핍",r"과잉\s*행동",
    r"주의력결핍과잉행동장애",r"집중력\s*문제",
    r"집중이\s*안",r"산만", r"멍때리",r"충동",
    r"약물\s*치료",r"콘서타", r"스트라테라"
]
WHITE_AD_REGEX = re.compile("|".join(ADHD_PATTERNS), re.IGNORECASE)



#나오면 거의 광고라고 봐도 되는 것들
HARD_AD_PATTERNS = [
    r"최저가", r"특가", r"특별가",r"쿠폰", r"공동구매", r"체험단",r"무료체험",
    r"무료상담",r"구매\s*문의", r"구매링크", r"쇼핑몰",
    r"프로모션",r"분양", r"투자", r"대출",r"부동산", r"청약",r"이벤트\s*(참여|진행)",
    r"문의\s*주세요", r"문의주세요",r"전화\s*주세요", r"전화주세요",
    r"연락\s*주세요",r"사마귀", r"내시경", r"휘트니스", r"피트니스", 
    r"비뇨기과", r"구강", r"접종", r"피부과",' r"한의원", r"시술", 
    r"보톡스", r"필러", r"여드름", r"탈모", r"모발이식",r"제공받", r"지원받", r"후원받",
    r"광고", r"협찬", r"피부과", r"한의원", r"시술", r"보톡스", r"필러", r"여드름", 
    r"탈모", r"모발이식",r"중개", r"산재",r"빠른상담", r"전화상담"
]

# 정상 글에도 등장할 수 있는 것들 (겹쳐 나올 때만 광고로 판단)
SOFT_AD_PATTERNS = [
    r"카톡", r"상담", r"도서", r"썰", r"영어", r"치아", r"자소서",
    r"복싱", r"캠프", r"육아", r"독서", r"후원", r"교육",r"학원",
    r"sponsored", r"\bAD\b", r"PR",r"단독",r"문의하기", r"카카오톡",
    r"톡톡",r"중개",r"레슨", r"강의", r"클래스", r"모집", r"채용",
    r"안내드립니다", r"진행하고 있습니다", r"빠르게 연락", r"상세내용"
]

HARD_AD_REGEX = re.compile("|".join(HARD_AD_PATTERNS))
SOFT_AD_REGEX = re.compile("|".join(SOFT_AD_PATTERNS))


# 2️⃣ 광고 포함 여부 판별 함수 (ADHD 화이트리스트 + HARD/SOFT 로직 반영)
def is_ad_post(text: str) -> bool:
    """
    광고성 글이면 True, 아니면 False 반환
    - HARD 패턴: 하나라도 매칭되면 거의 광고로 판단
    - SOFT 패턴: 여러 개 겹칠 때만 광고로 판단
    - ADHD 관련 키워드가 포함되어 있으면 웬만해서는 살려줌
    """
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    # ADHD 관련 내용이 있으면 웬만하면 살린다
    has_adhd = bool(ADHD_REGEX.search(text))

    # HARD 패턴 매칭 여부
    has_hard_ad = bool(HARD_AD_REGEX.search(text))

    # SOFT 패턴 몇 개나 걸리는지
    soft_hits = len(SOFT_AD_REGEX.findall(text))

    # 광고라고 판단하는 기준
    # 1) HARD 패턴이 있고, ADHD 언급이 전혀 없는 경우
    # 2) SOFT 패턴이 너무 많이 겹치는 경우(예: 2개 이상)이고, ADHD 언급이 없는 경우
    if has_adhd:
        # ADHD 관련 고민/경험 공유 글은 최대한 살려주기
        return False

    if has_hard_ad:
        return True

    if soft_hits >= 2:  # 필요하면 2 → 3으로 올리면 더 덜 지웁니다.
        return True

    return False




In [43]:
# 3️⃣ 광고글 판별 후 제거
total_df2["is_ad"] = total_df2["articles"].progress_apply(is_ad_post)
total_df2 = total_df2[total_df2["is_ad"] == False].reset_index(drop=True)
# (선택) 컬럼 정리
total_df2 = total_df2.drop(columns=["is_ad"])

#unnamed가 생기면 실행
#df = df.drop(columns=['Unnamed: 0'])
total_df2.info()

  0%|          | 0/79758 [00:00<?, ?it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79758 entries, 0 to 79757
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   date              73389 non-null  object
 1   content           68725 non-null  object
 2   contents          7558 non-null   object
 3   text              3475 non-null   object
 4   articles          79758 non-null  object
 5   article_words     79758 non-null  object
 6   re_article_words  79758 non-null  object
dtypes: object(7)
memory usage: 4.3+ MB


In [ ]:
# 3️⃣ 광고글 판별 후 제거
df2["is_ad"] = df2["articles"].progress_apply(is_ad_post)
df2 = df2[df2["is_ad"] == False].reset_index(drop=True)
# (선택) 컬럼 정리
df2 = df2.drop(columns=["is_ad"])

#unnamed가 생기면 실행
#df = df.drop(columns=['Unnamed: 0'])

In [10]:
## 너무많이 제거되거나 너무 안된거 같으면 PATTERN수정
df2.shape

(161129, 5)

## 문장제거

In [8]:



#원하는 문장제거
#아래같은 형식으로 추가해서 삭제 (GPT 사용 권장)
NOTICE_PAT = re.compile(
    r"""
    (http\S+|www\S+) |(\S+@\S+\.\S+) |(\d{8,}) |(게시판\s*주제와\s*맞지\s*않는\s*글?\s*작성\s*시?\s*이동\s*및?\s*삭제됩니다!?!*) |
    (광고성\s*특정\s*블러거\s*유튜버\s*SNS\s*url\s*기제\s*제재\s*됩니다!*) |
    (등업용\s*반복게시글\s*스티커\s*덧글도배\s*경고없이\s*강제탈퇴처리!*) |
    (게시판\s*주제와\s*맞지\s*않는\s*글.*이동.*삭제) |(광고성\s*(블로거|유튜버|SNS)?.*제재) |
    (등업용\s*반복게시글.*강제탈퇴) |(도배성\s*글.*삭제) |(욕설.*비방.*제재) |(운영원칙.*어긋나는.*삭제) |
    (개인정보.*노출.*삭제) |(정치.*종교.*제재) |(광고성.*(블로거|유튜버|SNS)?.*제재) |(홍보.*글.*금지) |
    (스팸.*차단) |(도배성\s*글.*삭제) |(반복게시글.*강제탈퇴) |(자동.*등록) |(네이버.*정책) |(서비스.*약관) |
    (로그인.*해주세요) |(신고.*처리) |(저작권.*침해) |(읽어주셔서.*감사) |(좋은\s*하루.*되세요) |(긴\s*글.*읽어주신)
    """,
    re.VERBOSE
)
tqdm.pandas()D
def remove_notice(text: str) -> str: 
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    cleaned = NOTICE_PAT.sub(" ", text)            # 문구 제거
    cleaned = re.sub(r"\s+", " ", cleaned).strip() # 공백 정리
    return cleaned

# 2) DataFrame에 적용 
#df["articles"] = df["articles"].progress_apply(remove_notice)







In [9]:
df2["articles"] = df2["articles"].progress_apply(remove_notice)

  0%|          | 0/161129 [00:00<?, ?it/s]

## 띄어쓰기 제거

In [10]:
#특문, 띄어쓰기 제거

def re_pattern(text):
    # 1) 제거할 특수문자만 선택 (이모지는 남기기 권장)
    text = re.sub(r"[^a-zA-Z가-힣0-9\s\.\?\!\,]", " ", text)

    # 2) 줄바꿈, 이모지, 감탄사, 말투는 그대로 유지
    # (불필요한 공백만 최소 정리)
    text = re.sub(r'\s{2,}', ' ', text).strip()

    return text

#df['articles'] = df['articles'].apply(re_pattern)  # ✅ 문자열에 적용


In [11]:
df2['articles'] = df2['articles'].apply(re_pattern)

## 토큰화

In [12]:
#본인 경로 지정 해서 사용

stopwords_df = pd.read_csv('./data/ko-stopwords.csv')

In [13]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
ALLOW_TAGS = {'NNG', 'NNP'}   # Kiwi 명사 태그
# Okt의 'Noun'과 동일한 명사 계열

def kiwi_tokenize(s: str, stopwords:set):
    tokens = []
    for token in kiwi.tokenize(s):
        if token.tag in ALLOW_TAGS and token.form not in stopwords:
            tokens.append(token.form)
    return tokens


In [69]:
tqdm.pandas()
# 토큰화 실행 (진행바 표시)
df['article_words'] = df['articles'].progress_apply(
    lambda x: okt_pos_tagging(x, stopwords_df)
)

  0%|          | 0/95885 [00:00<?, ?it/s]

In [14]:
tqdm.pandas()
# 토큰화 실행 (진행바 표시)
df2['article_words'] = df2['articles'].progress_apply(
    lambda x: kiwi_tokenize(x, stopwords_df)
)

  0%|          | 0/161129 [00:00<?, ?it/s]

In [15]:
df2 = df2[
    df2['articles'].fillna('').str.strip().str.len() > 10
].reset_index(drop=True)

In [93]:
df2.shape

(117898, 5)

In [16]:
df2.to_csv('./data/약한광고제거_토큰화완료.csv',encoding='utf-8-sig')

## 불용어 사전 (원하는 단어 제거)

In [17]:
stopwords = set(stopwords_df['stopwords'])

ad_stopwords = [
    "카톡", "상담", "문의", "연락", "클릭", "전화", "광고", "협찬",
    "제공받", "지원받", "후원", "프로모션", "이벤트", "체험단",
    "특가", "할인", "쿠폰", "공동구매", "특별가",
    "무료", "무료상담", "쇼핑몰", "구매", "구매링크",
    "톡톡", "카카오톡", "상세내용", "진행중", "진행하고",
]
medical_stopwords = [
    "치과", "피부과", "한의원", "시술", "보톡스",
    "필러", "내시경", "사마귀", "이식", "탈모",
]
irrelevant_stopwords = [
    "영어", "도서", "독서", "육아", "과외", "강의", "레슨",
    "스터디", "클래스", "캠프", "피트니스", "휘트니스", "복싱",
]
korean_filler = [
    "그리고", "그러나", "근데", "그냥", "그래서", "음", "어", "아",
    "정말", "진짜", "약간", "좀", "너무", "많이", "사실", "저는",
    "제", "저", "저희", "우리", "그", "이", "저런", "이런",
    "그런", "거", "것", "하는", "하는데", "합니다", "했어요",
    "습니다", "있어요", "있습니다",
]
platform_noise = [
    "블로그", "카페", "포스트", "댓글", "작성자", "조회수",
    "사진", "이미지", "링크", "url", "www", "http",
]

custom_stopwords = list(set(
    ad_stopwords
    + medical_stopwords
    + irrelevant_stopwords
    + korean_filler
    + platform_noise
))


# 리스트 안에 사용할 불용어 추가
stopwords.update(custom_stopwords)

In [18]:
df2.head()

,date,content,contents,text,articles
0,2023. 9. 1. 10:25,#성인ADHD\n#성인ADHD자가진단\n나는 왜 이렇게 집중을 못할까?\n나는 왜 ...,NaN,NaN,#성인ADHD\n#성인ADHD자가진단\n나는 왜 이렇게 집중을 못할까?\n나는 왜 ...
1,2023. 8. 29. 16:13,성인 ADHD 테스트 방법 원인 유전 있을까요?\n성인ADHD\n성인 ADHD는 인...,NaN,NaN,성인 ADHD 테스트 방법 원인 유전 있을까요?\n성인ADHD\n성인 ADHD는 인...
2,2023. 8. 25. 15:47,"성인 ADHD 원인, 증상 그리고 자가진단 방법\n성인 ADHD는 주의력 결핍 과잉...",NaN,NaN,"성인 ADHD 원인, 증상 그리고 자가진단 방법\n성인 ADHD는 주의력 결핍 과잉..."
3,2023. 8. 24. 11:50,#김해성인ADHD상담#김해심리상담센터#김해우울증상담#김해조울증상담#김해불안증상담#김...,NaN,NaN,#김해성인ADHD상담#김해심리상담센터#김해우울증상담#김해조울증상담#김해불안증상담#김...
4,2023. 8. 22. 19:53,주위에 보면 꼭 주의력이 안 좋은 친구들이 하나둘씩 있죠? 친구뿐만 아니라 가족들 ...,NaN,NaN,주위에 보면 꼭 주의력이 안 좋은 친구들이 하나둘씩 있죠? 친구뿐만 아니라 가족들 ...


In [20]:
## 1글자 토큰 제거 함수 
def remove_short_tokens(tokens, min_len=2):
    return [t for t in tokens if len(t) >= min_len]

# 적용 예        
#df["re_article_words"] = df["article_words"].progress_apply(lambda x: remove_short_tokens(x, min_len=2))

In [21]:
df2["re_article_words"] = df2["article_words"].progress_apply(lambda x: remove_short_tokens(x, min_len=2))

  0%|          | 0/121528 [00:00<?, ?it/s]

## 2차 불용어사전 (빈도수가 낮은 관계없는글 제거)

In [22]:
from collections import Counter
all_tokens = [token for tokens in df2["re_article_words"] for token in tokens]

token_counts = Counter(all_tokens)

rare_tokens = {word: cnt for word, cnt in token_counts.items() if cnt < 10}


In [23]:
# 3) 빈도가 10 미만인 단어만 필터링, 글이나 광고 토큰이 너무 안지워지면 숫자 늘리기 
rare_tokens = {word: cnt for word, cnt in token_counts.items() if cnt < 10}

print(len(rare_tokens))        # 몇 개인지 확인
print(list(rare_tokens.items())[:50])

184490
[('업앤', 4), ('스튜머', 1), ('하이퍼엑티비티', 1), ('임퓨시브', 5), ('하녹', 1), ('마족', 2), ('클로나제팜과', 1), ('유백색', 7), ('스파톤', 4), ('저룩', 1), ('인사부', 8), ('저떻고', 3), ('후버링', 9), ('힐러세연', 9), ('연제동', 2), ('비스무트', 3), ('비스텔라민', 3), ('어버버버버', 1), ('스몰톡', 8), ('외딴섬', 6), ('스트레쓰를', 1), ('리아킴오', 1), ('채움테라스', 1), ('통진', 6), ('존스워트', 1), ('히페리신', 7), ('히퍼포린', 6), ('김봉수', 6), ('파크포레', 4), ('트리지움', 7), ('파크리오', 3), ('그라시움', 2), ('잊어버', 4), ('뇌음탄', 5), ('연에', 1), ('터스트', 1), ('유길상', 7), ('칼럽', 1), ('에라이', 9), ('저존검아', 1), ('장복심', 1), ('창우', 5), ('임팔', 7), ('해찰', 5), ('유소아기', 4), ('2순환로', 4), ('처피', 6), ('띄엄', 6), ('뜨엥', 1), ('에스더버니', 1)]


In [28]:
# 희귀 토큰(rare_tokens)을 stopwords에 추가
stopwords.update(rare_tokens.keys())

THRESH = 0.10  # 불용어 비율 15% 이상이면 삭제 (상황에 맞게 조정)  0.15 면 이미 높은 수준이지만 0.1 까지는 가능 


def stop_ratio(tokens):
    if not tokens: return 1.0
    s = sum(1 for w in tokens if w in stopwords)
    return s / len(tokens)

mask = df2["re_article_words"].apply(lambda t: stop_ratio(t) < THRESH)
df_filtered = df2[mask].copy()

In [29]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 98956 entries, 0 to 121527
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   date              89675 non-null  object
 1   content           78289 non-null  object
 2   contents          15182 non-null  object
 3   text              5485 non-null   object
 4   articles          98956 non-null  object
 5   article_words     98956 non-null  object
 6   re_article_words  98956 non-null  object
dtypes: object(7)
memory usage: 6.0+ MB


## 코사인 유사도 기반 중복글 제거 

In [31]:
def ko_ratio(tokens):
    j = "".join(tokens)
    return 0 if not j else sum("가" <= ch <= "힣" for ch in j) / len(j)
    
# 0) ko_ratio 한 번만 계산
df_filtered = df_filtered.assign(
    _len = df_filtered["re_article_words"].str.len(),
    _ko  = df_filtered["re_article_words"].progress_apply(ko_ratio)
)

# 1) 베이스 마스크
# 토큰수가 5개 이상/ 한글 비율이 40% 이상만 남김
base_mask = (df_filtered["_len"] >= 3) & (df_filtered["_ko"] >= 0.4)  
idx_base  = df_filtered.index[base_mask]
corpus    = df_filtered.loc[idx_base, "re_article_words"].tolist()

# TF-IDF (identity tokenizer)
vec = TfidfVectorizer(tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None,
                      min_df=3, max_df=0.9)
X = vec.fit_transform(corpus)   # sparse CSR

#반경 이웃 탐색 (코사인 거리)
from sklearn.neighbors import NearestNeighbors
thr = 0.05  # = 1 - 0.95
nn  = NearestNeighbors(metric="cosine", algorithm="brute").fit(X)
dists_list, neighs_list = nn.radius_neighbors(X, radius=thr, return_distance=True)

#앞에서 하나만 남기기
kept_local = np.ones(len(idx_base), dtype=bool)
for i, neighs in tqdm(enumerate(neighs_list), total=len(neighs_list)):
    if not kept_local[i]:
        continue
    # 자기 자신 포함되어 있으므로 i보다 큰 이웃만 제거
    for j in neighs:
        if j > i:
            kept_local[j] = False

keep = pd.Series(False, index=df_filtered.index)
keep.loc[idx_base] = kept_local

# 5) 최종
total_df = df_filtered.loc[keep].drop(columns=["_len","_ko"]).copy()


  0%|          | 0/93556 [00:00<?, ?it/s]

In [32]:
total_df = total_df.reset_index(drop=True)

In [33]:
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79944 entries, 0 to 79943
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   date              73554 non-null  object
 1   content           68839 non-null  object
 2   contents          7629 non-null   object
 3   text              3476 non-null   object
 4   articles          79944 non-null  object
 5   article_words     79944 non-null  object
 6   re_article_words  79944 non-null  object
dtypes: object(7)
memory usage: 4.3+ MB


In [34]:
total_df['articles'].duplicated().sum()


np.int64(0)

In [35]:
total_df

,date,content,contents,text,articles,article_words,re_article_words
0,2023. 9. 1. 10:25,#성인ADHD\n#성인ADHD자가진단\n나는 왜 이렇게 집중을 못할까?\n나는 왜 ...,NaN,NaN,성인ADHD 성인ADHD자가진단 나는 왜 이렇게 집중을 못할까? 나는 왜 시작하는건...,"[성인, 성인, 자, 진단, 집중, 시작, 마무리, 사람, 한번쯤, 생각, 경우, ...","[성인, 성인, 진단, 집중, 시작, 마무리, 사람, 한번쯤, 생각, 경우, 부분,..."
1,2023. 8. 29. 16:13,성인 ADHD 테스트 방법 원인 유전 있을까요?\n성인ADHD\n성인 ADHD는 인...,NaN,NaN,성인 ADHD 테스트 방법 원인 유전 있을까요? 성인ADHD 성인 ADHD는 인지하...,"[성인, 테스트, 방법, 원인, 유전, 성인, 성인, 인지, 때, 스스로, 병, 인...","[성인, 테스트, 방법, 원인, 유전, 성인, 성인, 인지, 스스로, 인지, 치료,..."
2,2023. 8. 25. 15:47,"성인 ADHD 원인, 증상 그리고 자가진단 방법\n성인 ADHD는 주의력 결핍 과잉...",NaN,NaN,"성인 ADHD 원인, 증상 그리고 자가진단 방법 성인 ADHD는 주의력 결핍 과잉행...","[성인, 원인, 증상, 자가, 진단, 방법, 성인, 주의력 결핍 과잉행동장애, 약자...","[성인, 원인, 증상, 자가, 진단, 방법, 성인, 주의력 결핍 과잉행동장애, 약자..."
3,2023. 8. 22. 19:53,주위에 보면 꼭 주의력이 안 좋은 친구들이 하나둘씩 있죠? 친구뿐만 아니라 가족들 ...,NaN,NaN,주위에 보면 꼭 주의력이 안 좋은 친구들이 하나둘씩 있죠? 친구뿐만 아니라 가족들 ...,"[주위, 주의력, 친구, 친구, 가족, 집중, 불편, 적, 학원, 반복, 인강, 요...","[주위, 주의력, 친구, 친구, 가족, 집중, 불편, 학원, 반복, 인강, 요새, ..."
4,2023. 8. 31. 20:50,평소에도 워낙 너무 당연한 것도\n잘 까먹고! 빼먹고! 잊어버리고!\n그러다 집 근...,NaN,NaN,평소에도 워낙 너무 당연한 것도 잘 까먹고! 빼먹고! 잊어버리고! 그러다 집 근처 ...,"[평소, 당연, 집, 근처, 정신과, 때, 의사, 선생, 남편, 처음, 말씀, 성인...","[평소, 당연, 근처, 정신과, 의사, 선생, 남편, 처음, 말씀, 성인, 검사, ..."
...,...,...,...,...,...,...,...
79939,2024.10.14,NaN,제가 저번달부터 주변 환경들이 많이 바뀌면서 적응해야할게 굉장히 많았어요. 제가 선...,NaN,제가 저번달부터 주변 환경들이 많이 바뀌면서 적응해야할게 굉장히 많았어요. 제가 선...,"[저번, 달, 주변, 환경, 적응, 선택, 길, 감당, 악바리, 자존감, 일, 기력...","[저번, 주변, 환경, 적응, 선택, 감당, 악바리, 자존감, 기력, 컨트롤, 상황..."
79940,2024.10.08,NaN,너무 무기력하고 이미 지각했는데 그냥 결석할까 생각도들고요 저번주에도 이랬는데 그냥...,NaN,너무 무기력하고 이미 지각했는데 그냥 결석할까 생각도들고요 저번주에도 이랬는데 그냥...,"[기력, 지각, 결석, 생각, 저번주, 학교, 무단결석]","[기력, 지각, 결석, 생각, 저번주, 학교, 무단결석]"
79941,2024.09.19,NaN,"저는 스무살이고, 지인은 고등학생이에요. 지인은 가정사가 좀 어둡고, 지금도 그닥 ...",NaN,"저는 스무살이고, 지인은 고등학생이에요. 지인은 가정사가 좀 어둡고, 지금도 그닥 ...","[지인, 고등학생, 지인, 가정사, 지금, 녹, 록, 병원, 말, 정도, 애, 남,...","[지인, 고등학생, 지인, 가정사, 지금, 병원, 정도, 피해, 이야기, 묘사, 진..."
79942,2024.09.27,NaN,사실 최근에 문제가 해결됬습니다만... 걱정이 되어서 글 적어봅니다.근 몇달간 아침...,NaN,사실 최근에 문제가 해결됬습니다만... 걱정이 되어서 글 적어봅니다.근 몇달간 아침...,"[최근, 문제, 해결, 걱정, 글, 아침, 기상, 알람, 소리, 친구, 집, 날, ...","[최근, 문제, 해결, 걱정, 아침, 기상, 알람, 소리, 친구, 휴대폰, 알람, ..."


In [36]:
total_df2 = total_df.copy()

In [37]:
total_df.to_csv('./data/전처리완료.csv',encoding = 'utf-8-sig')